In [1]:
import utils
import pandas as pd
from pathlib import Path
from collections import defaultdict 
import json
import matplotlib.pyplot as plt
from itertools import combinations

In [2]:
# make a dictionary with key = drug, values = targets
drug_table = pd.read_excel(Path("../data/drugs_with_targets.xlsx"))

drug_t = (
    drug_table
        .dropna(subset=[drug_table.columns[2], drug_table.columns[3]])
        .assign(Genes=lambda df: df.iloc[:, 3].str.split(";"))
        .explode("Genes")
        .assign(Genes=lambda df: df["Genes"].str.strip())
        .groupby(drug_table.columns[2])["Genes"]
        .unique()
        .apply(list)
        .to_dict()
)

In [3]:
# transform into a dictionary where for each gene we have the corresponding drugs
dict_genes = defaultdict(set)

for drug, genes_list in drug_t.items():
    for genes in genes_list:
        for gene in genes.split(","):
            gene = gene.strip()
            if gene:
                dict_genes[gene].add(drug)

dict_genes = {g: ",".join(sorted(d)) for g, d in dict_genes.items()}

In [4]:
# read patients lists (primary, secondary, and all for primary)
patient_ids = pd.read_csv("../data/PatientIds.txt", header=None).iloc[:, 0].tolist()

In [5]:
#function to extract the filename and the run
def parse_filename(f):
    
    if len(f) >= 38: 
        part1 = f[:36]
        part2 = f[37:]
    else: 
        part1 = f[:3]
        part2 = f[4:]

    return part1, part2

In [6]:
# create a dictionary with key=cases, and values contains information like run, control nodes, proteins controlled, drugs
# filter the .json data to remain only with the data where control nodes that have drugs which act on them and can control more than two proteins

def analysis_results(path):

    row_dict = defaultdict(list)

    for filename in path.glob("*.json"):
        with open(filename, "r") as f:
            content = json.load(f)

        target_to_proteins = defaultdict(set)

        for solution in content.get("Solutions", {}).get("Solutions", []):
            for path_entry in solution.get("Paths", []):
                source = path_entry[0]
                target = path_entry[-1]
                target_to_proteins[source].add(target)
            
        just_filename = filename.stem
        file_id, file_label = parse_filename(just_filename)

        for gene, targets in target_to_proteins.items():
            if gene in dict_genes and len(targets) > 2:
                targets_str = " , ".join(sorted(targets))
                row_dict[file_id].append(
                    (
                        file_id,
                        file_label,
                        gene,
                        targets_str,
                        len(targets),
                        dict_genes[gene],
                    )
                )
    return row_dict


res_withoutSTRING = analysis_results(Path("../data/database_exclusion_results/AnalysesFromUpWithoutDown-InBetween1_withoutSTRING"))
res_withoutSIGNOR = analysis_results(Path("../data/database_exclusion_results/AnalysesFromUpWithoutDown-InBetween1_withoutSIGNOR"))
res_withoutOmniPath = analysis_results(Path("../data/database_exclusion_results/AnalysesFromUpWithoutDown-InBetween1_withoutOmniPath"))
res_withoutInnateDB = analysis_results(Path("../data/database_exclusion_results/AnalysesFromUpWithoutDown-InBetween1_withoutInnateDB"))
res_withoutKEGG = analysis_results(Path("../data/database_exclusion_results/AnalysesFromUpWithoutDown-InBetween1_withoutKEGG"))
res_gap0=analysis_results(Path("../data/AnalysesFromUpWithoutDown-InBetween1-MaxPath3gap0"))
res_gap2=analysis_results(Path("../data/AnalysesFromUpWithoutDown-InBetween1-MaxPath3gap2"))
res_gap3=analysis_results(Path("../data/AnalysesFromUpWithoutDown-InBetween1-MaxPath3gap3"))
res_path4=analysis_results(Path("../data/AnalysesFromUpWithoutDown-InBetween1-MaxPath4"))
res_path5=analysis_results(Path("../data/AnalysesFromUpWithoutDown-InBetween1-MaxPath4_path5"))
res_same_seed = analysis_results( Path("../data/AnalysesFromUpWithoutDown-InBetween1_seed587548125"))
res_again = analysis_results(Path("../data/AnalysesFromUpWithoutDown-InBetween1_again"))
res_paper =  analysis_results(Path("../data/AnalysesFromUpWithoutDown-InBetween1-MaxPath3"))

In [7]:
# transform the dictionary so that we have separate information for each drug in each case
def patient_drug_p(patient_list, drug_t, row_dict):
    
    patient_drug_protein = defaultdict(list)
    drug_set = set(drug_t.keys())
    
    for patient, rows in row_dict.items():
        for row in rows:
            drugs_in_row = {d.strip() for d in row[5].split(",")}

            for drug in drug_set & drugs_in_row:
                patient_drug_protein[patient].append(
                    (
                        patient,
                        drug,
                        row[1],
                        row[2],
                        row[3],
                        #row[4]
                    )
                )

    return patient_drug_protein
patient_res_STRING = patient_drug_p(utils.patient_ids,drug_t,res_withoutSTRING)
patient_res_SIGNOR = patient_drug_p(utils.patient_ids,drug_t,res_withoutSIGNOR)
patient_res_OmniPath = patient_drug_p(utils.patient_ids,drug_t,res_withoutOmniPath)
patient_res_InnateDB = patient_drug_p(utils.patient_ids,drug_t,res_withoutInnateDB)
patient_res_KEGG = patient_drug_p(utils.patient_ids,drug_t,res_withoutKEGG)
patient_res_gap0=patient_drug_p(utils.patient_ids,drug_t,res_gap0)
patient_res_gap2=patient_drug_p(utils.patient_ids,drug_t,res_gap2)
patient_res_gap3=patient_drug_p(utils.patient_ids,drug_t,res_gap3)
patient_res_path4=patient_drug_p(utils.patient_ids,drug_t,res_path4)
patient_res_path5=patient_drug_p(utils.patient_ids,drug_t,res_path5)
patient_res_same_seed=patient_drug_p(utils.patient_ids,drug_t,res_same_seed)
patient_res_again = patient_drug_p(utils.patient_ids,drug_t,res_again)
patient_res_paper = patient_drug_p(utils.patient_ids,drug_t, res_paper)

In [8]:
#see the overlap when the seed doesn't change
rows_seed = []

for lista in patient_res_same_seed.values():
    for t in lista:
        rows_seed.append([t[0], t[2], t[1]])

df_same_seed = pd.DataFrame(rows_seed, columns=["case", "run", "drug"])
df_same_seed  = df_same_seed .drop_duplicates()
n_analize = df_same_seed ["run"].nunique()

freq = (
    df_same_seed .groupby(["case", "drug"])["run"]
      .nunique()
      .reset_index(name="aparitii")
)

freq["procent"] = 100 * freq["aparitii"] / n_analize
freq = freq.sort_values(["case", "procent"], ascending=[True, False])

print("If we don't modify the seed, the minimum percentage in which a drug appears is:",freq['procent'].min())

If we don't modify the seed, the minimum percentage in which a drug appears is: 100.0


In [9]:
#see the mean number of runs and the standard deviation
def mean_runs(patient_res):
    df = pd.DataFrame(
        [t for lst in patient_res.values() for t in lst],columns=["Case","Drug","Run","Control","Essential proteins"]
    )

    resultat = (
        df.groupby("Case")
          .apply(lambda g: g.loc[~g["Drug"].duplicated(), "Run"].iloc[-1])
          .reset_index(name="Last_drug")
    )
    resultat["Last_drug"] = pd.to_numeric(resultat["Last_drug"], errors="coerce")

    m = resultat["Last_drug"].mean()
    std = resultat["Last_drug"].std()
    print("The mean for last appearance: ",m)
    print("The standard deviation for last appearance: ",std)

mean_runs(patient_res_paper)
mean_runs(patient_res_again)

The mean for last appearance:  85.96026490066225
The standard deviation for last appearance:  20.80156429845345
The mean for last appearance:  85.23178807947019
The standard deviation for last appearance:  22.130053082813053


In [10]:
#function to format strings and lists
def to_list_clean(x):
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        return [v.strip() for v in x.split(",") if v.strip()]
    return []
    
# generate per-patient drug reports where each drug appears once per patient; control nodes and patient proteins are aggregated across all runs
def report_one_patient(patient_drug_protein):
    
    summary_df = pd.DataFrame()
    
    for file_id, records in patient_drug_protein.items():

        df = pd.DataFrame(
                records,
                columns=["Patient", "Drug", "Run", "Control nodes", "Patient proteins"]
            )

        # group so that each Drug appears once per Patient
        grouped = (
                df.groupby(["Patient", "Drug"], as_index=False)
                      .agg({
                      "Patient proteins": lambda x: sum((to_list_clean(v) for v in x), []),
                      "Control nodes": lambda x: sum((to_list_clean(v) for v in x), []),
                      }))

        # convert lists to comma-separated strings
        grouped["Patient proteins"] = (
                grouped["Patient proteins"]
                .apply(lambda x: ",".join(map(str, set(x))))
                        )

        grouped["Control nodes"] = (
                grouped["Control nodes"]
                .apply(lambda x: ",".join(map(str, set(x))))
            )

        # the number of controlled proteins and control nodes
        grouped["Patient proteins length"] = (
                grouped["Patient proteins"].str.split(",").str.len()
            )
        grouped["Control nodes length"] = (
                grouped["Control nodes"].str.split(",").str.len()
            )

        # number of runs per drug            
        grouped["No of runs"] = (
                df.groupby(["Patient", "Drug"])["Run"]
                .nunique()
                .values
            )

        summary_df = pd.concat([summary_df, grouped], ignore_index=True)

        

    return summary_df
    
summary_df_STRING = report_one_patient(patient_res_STRING)
summary_df_SIGNOR = report_one_patient(patient_res_SIGNOR)
summary_df_KEGG = report_one_patient(patient_res_KEGG)
summary_df_OmniPath = report_one_patient(patient_res_OmniPath)
summary_df_InnateDB = report_one_patient(patient_res_InnateDB)
summary_df_gap0 = report_one_patient(patient_res_gap0)
summary_df_gap2 = report_one_patient(patient_res_gap2)
summary_df_gap3 = report_one_patient(patient_res_gap3)
summary_df_path4 = report_one_patient(patient_res_path4)
summary_df_path5 = report_one_patient(patient_res_path5)
summary_df_paper=report_one_patient(patient_res_paper)

In [11]:
#find the proportion in which the drugs included in the results presented are in variants where path=4 or path = 5
def comparison_path_included(summary_df_path,nr):
    
    comparison = summary_df_paper.merge(
        summary_df_path[['Patient', 'Drug']],
        on=['Patient', 'Drug'],
        how='left',
        indicator=True
    )

    result_path = (
        comparison.groupby('Patient')
        .agg(
            sum_drugs=('Drug', 'count'),
            common_val=('_merge', lambda x: (x == 'both').sum())
        )
    )

    result_path['proportion'] = result_path['common_val'] / result_path['sum_drugs']

    m=result_path['proportion'].mean()
    s=result_path['proportion'].std()
    print("Mean of results included in path equal to",nr,"is", m," and std=",s)

comparison_path_included(summary_df_path4,"4")
comparison_path_included(summary_df_path5,"5")

Mean of results included in path equal to 4 is 0.900324752034432  and std= 0.06620635732169138
Mean of results included in path equal to 5 is 0.9314829751414051  and std= 0.057988470253336796


In [12]:
def jaccard(a, b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

In [13]:
# function to calculate jaccard index between results:
def calculate_jaccard_for_dataftames(dfs,names):
    col1 = "Patient"
    col2 = "Drug"

    grouped = [
        df.groupby(col1)[col2].apply(set).to_dict()
        for df in dfs
    ]

    all_keys = set().union(*(d.keys() for d in grouped))

    rows = []

    pairs = list(combinations(range(len(grouped)), 2))

    for key in all_keys:
        row = {col1: key}

        scores = []

        for i, j in pairs:
            score = jaccard(
                grouped[i].get(key, set()),
                grouped[j].get(key, set())
            )

            row[f"{names[i]} vs {names[j]}"] = score
            scores.append(score)

        row["mean"] = sum(scores) / len(scores)
        row["std"] = pd.Series(scores).std()

        rows.append(row)

    results = pd.DataFrame(rows)

    pair_cols = [c for c in results.columns if " vs " in c]

    summary = pd.DataFrame({
        "mean": results[pair_cols].mean(),
        "std": results[pair_cols].std()
    })

    print(summary)

In [14]:
# compare results on networks with different gaps
calculate_jaccard_for_dataftames([summary_df_paper, summary_df_gap2, summary_df_gap0,summary_df_gap3],["Paper", "Gap2","Gap0", "Gap3"])


                   mean       std
Paper vs Gap2  0.717694  0.076243
Paper vs Gap0  0.228796  0.079876
Paper vs Gap3  0.718320  0.077132
Gap2 vs Gap0   0.229161  0.081614
Gap2 vs Gap3   0.740901  0.077253
Gap0 vs Gap3   0.231204  0.084770


In [15]:
# compare results on networks with different path length
calculate_jaccard_for_dataftames([summary_df_path4,summary_df_path5, summary_df_paper], ["Path_4","Path_5","Paper"])

                      mean       std
Path_4 vs Path_5  0.718831  0.067734
Path_4 vs Paper   0.618520  0.071913
Path_5 vs Paper   0.554841  0.062505


In [16]:
# compare results on networks obtained by exclusion of one database
calculate_jaccard_for_dataftames([summary_df_InnateDB, summary_df_OmniPath, summary_df_SIGNOR, summary_df_STRING, summary_df_KEGG],["InnateDB","OmniPath","SIGNOR","STRING","KEGG"])

                          mean       std
InnateDB vs OmniPath  0.502918  0.058096
InnateDB vs SIGNOR    0.700079  0.083545
InnateDB vs STRING    0.453755  0.056827
InnateDB vs KEGG      0.704104  0.085593
OmniPath vs SIGNOR    0.509585  0.065048
OmniPath vs STRING    0.428877  0.062520
OmniPath vs KEGG      0.504464  0.060343
SIGNOR vs STRING      0.422063  0.059210
SIGNOR vs KEGG        0.715591  0.084708
STRING vs KEGG        0.441866  0.053985
